# Four judges score ten startup pitches: ICC

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Intraclass Correlation Coefficient** · a worked example from the StatNex teaching guide

**The story.** Four judges scored ten student startup pitches out of 30 at a campus competition. The organisers want to know whether the judges are consistent enough to trust the ranking.

**The question.** Do the judges agree on which pitches are good, and does any judge score differently from the others?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `01_basic/intraclass_correlation.csv` (10 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Startup,Judge 1,Judge 2,Judge 3,Judge 4
EcoCup,12.0,7.0,18.0,11.0
StudyBuddy,17.0,17.0,21.0,18.0
PetPal,4.0,6.0,9.0,5.0
FreshBox,12.0,9.0,13.0,9.0
CampusRide,16.0,16.0,19.0,18.0
SolarSnack,20.0,25.0,27.0,22.0
CodeKids,4.0,5.0,5.0,5.0
ThriftLoop,8.0,9.0,12.0,10.0
MealMate,20.0,19.0,26.0,23.0
FitTrack,13.0,10.0,14.0,9.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm

pd.set_option("display.width", 140)

## The data

In [ ]:
# One row per subject, one column per rater. Only subjects scored by every rater are
# used.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
raters = ["Judge 1", "Judge 2", "Judge 3", "Judge 4"]
scores = data[raters].apply(pd.to_numeric, errors="coerce")
complete = scores.dropna()

## Case summary

In [ ]:
total, valid = len(scores), len(complete)
cases = pd.DataFrame({"n": [valid, total - valid, total],
                      "share": [valid / total, (total - valid) / total, 1.0]},
                     index=["valid", "incomplete", "total"])
print(cases)

## Analysis of variance table

In [ ]:
# A two-way layout without replication: subjects × raters, one score per cell.
# statsmodels fits score ~ subject + rater by least squares; anova_lm() splits the
# variation into subjects (how much the items really differ), raters (how much the
# raters' average levels differ) and the residual (disagreement left over). The
# raters' F tests whether some rater scores systematically higher or lower.
subjects, k = complete.shape
long = pd.DataFrame({"score": complete.to_numpy(float).ravel(order="F"),
                     "subject": np.tile(np.arange(subjects), k),
                     "rater": np.repeat(np.arange(k), subjects)})
fit = smf.ols("score ~ C(subject) + C(rater)", data=long).fit()
table = anova_lm(fit)
anova = pd.DataFrame(
    table[["sum_sq", "df", "mean_sq", "F", "PR(>F)"]].to_numpy(),
    index=["between subjects", "between raters", "residual"],
    columns=["SS", "df", "MS", "F", "P"],
)
anova.loc["total"] = [table["sum_sq"].sum(), table["df"].sum(), np.nan, np.nan, np.nan]
ms_subject, ms_rater, ms_error = anova["MS"].iloc[:3]
df_subject, df_rater, df_error = anova["df"].iloc[:3]
f_subject, p_subject = anova.loc["between subjects", ["F", "P"]]

## Intraclass correlation

In [ ]:
# Shrout & Fleiss's case 3 (two-way mixed, consistency): the share of the score
# variation that is real difference between subjects, once each rater's own average
# level is set aside. ICC(3,1) is the reliability of one rater's score, ICC(3,k) that
# of the mean of all k raters. A rater who is generous to everyone by the same amount
# does not count against it. It assumes the scores are roughly normal with the same
# spread for every subject. The confidence limits come from the F distribution of the
# subjects' F ratio; the P value tests "ICC = 0", which a usable scale passes easily,
# so judge the ICC and its lower limit (above 0.75 good, above 0.9 excellent, in Koo
# and Li's widely used guideline). There is no ICC function in SciPy or statsmodels,
# so it is computed from the ANOVA's mean squares.
level = 95 / 100
alpha = 1 - level
lower_f = f_subject / stats.f.ppf(1 - alpha / 2, df_subject, df_error)
upper_f = f_subject * stats.f.ppf(1 - alpha / 2, df_error, df_subject)
single = (ms_subject - ms_error) / (ms_subject + (k - 1) * ms_error)
average = (ms_subject - ms_error) / ms_subject
icc = pd.DataFrame(
    [[single, (lower_f - 1) / (lower_f + k - 1), (upper_f - 1) / (upper_f + k - 1),
      f_subject, df_subject, df_error, p_subject],
     [average, (lower_f - 1) / lower_f, (upper_f - 1) / upper_f,
      f_subject, df_subject, df_error, p_subject]],
    index=["ICC(3,1)", f"ICC(3,{k})"],
    columns=["ICC", "lower", "upper", "F", "df1", "df2", "P"],
)
print()
print(icc)
print()
print(anova)

## Absolute agreement (not in the app)

In [ ]:
# ICC(A,1), McGraw & Wong's absolute agreement for one rater (Shrout & Fleiss's
# ICC(2,1)): it also counts a generous rater's offset as disagreement, so it is the
# one that matters when a fixed cut-off, such as a pass mark, is applied to single
# scores. It is lower than ICC(3,1) whenever the raters' F above is large.
agreement = (ms_subject - ms_error) / (
    ms_subject + (k - 1) * ms_error + k * (ms_rater - ms_error) / subjects)
print(f"\nICC(A,1) = {agreement:.4f}")

## What the result says

The intraclass correlation (ICC) measures how well raters agree. A value of 1 is perfect agreement. For one judge alone, ICC(3,1) is 0.93 (95% confidence interval, the range of values the data agree with for the true value: 0.83 to 0.98). For the average of the four judges, ICC(3,4) is 0.98. So the judges agree very well on which pitches are better, and the ranking can be trusted.

The analysis of variance table adds one warning. The row between raters has F = 11.8. F compares the differences between the judges' average scores with the leftover variation. Its P value, the chance of an F this large if the judges scored alike on average, is very small. So the judges do not give the same number of points on average. The data show who: Judge 3 gave the highest score, or shared it, for every pitch.

Trust the ranking, but do not compare raw points between judges.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.